# GaussCapture Colab Trainer

Use a CUDA GPU runtime. TPU is not supported. Upload `dataset.zip` from GaussCapture or point to a Drive path.

In [ ]:
import os, json, zipfile, shutil, subprocess, pathlib
from google.colab import files
!nvidia-smi

## Upload Dataset

In [ ]:
uploaded = files.upload()
dataset_zip = next(iter(uploaded.keys()))
work = pathlib.Path('/content/gausscapture_dataset')
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile(dataset_zip, 'r') as z:
    z.extractall(work)
print('Extracted to', work)

## Configure Trainer

In [ ]:
TRAINER_REPO = 'https://github.com/graphdeco-inria/gaussian-splatting.git'
trainer = pathlib.Path('/content/gaussian-splatting')
if not trainer.exists():
    !git clone --recursive {TRAINER_REPO} /content/gaussian-splatting
print('Trainer path:', trainer)

## Read Config and Train

In [ ]:
config_path = work / 'colab_export' / 'run_config.json'
config = json.loads(config_path.read_text()) if config_path.exists() else {'preset':'draft'}
iterations = {'draft': 5000, 'balanced': 15000, 'high': 30000}.get(config.get('preset'), 5000)
output = pathlib.Path('/content/gausscapture_output')
output.mkdir(exist_ok=True)
cmd = ['python', 'train.py', '-s', str(work), '-m', str(output), '--iterations', str(iterations)]
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(trainer), check=True)

## Package Result

In [ ]:
result_zip = '/content/gausscapture_training_result.zip'
shutil.make_archive('/content/gausscapture_training_result', 'zip', output)
files.download(result_zip)